# PrimaTap Dictionary Builder
Uses Claude to decompose the most common English words into semantic primitives.
Outputs a lookup table: word → primitive sequence.

In [ ]:
import anthropic
import json
import time
import os
from wordfreq import top_n_list
from pathlib import Path

# --- Primitive definitions ---
PRIMITIVES = [
    'SELF',   # I, me, my, mine — the communicating agent
    'OTHER',  # you, they, it, someone else — any non-SELF entity
    'NOT',    # negation, absence, opposite
    'WANT',   # desire, need, intention, wish
    'GOOD',   # positive, correct, safe, pleasant
    'BAD',    # negative, wrong, dangerous, unpleasant
    'NOW',    # present time, immediate, current
    'HERE',   # this place, location, spatial reference
    'ALIVE',  # living, active, functioning, conscious
    'MOVE',   # motion, action, change, go
    'WATER',  # liquid, fluid
    'AIR',    # gas, breath, sky, wind
    'HEAT',   # temperature, energy, warmth, fire
    'SOLID',  # physical object, earth, matter, hard
    'LIGHT',  # visible energy, brightness, color, see
]

PRIMITIVE_SET = set(PRIMITIVES)
print(f'Primitives: {PRIMITIVES}')

In [ ]:
# --- System prompt (will be cached) ---
SYSTEM_PROMPT = f"""You are building a dictionary for PrimaTap, a communication system based on semantic primitives.

The 15 primitives and their meanings:
{chr(10).join(f'  {p}' for p in PRIMITIVES)}

For each English word given, output a JSON object with:
  "word": the input word
  "primitives": a list of 1-4 primitives that best capture its core meaning
  "note": one short phrase explaining the decomposition

Rules:
- Only use primitives from the list above, spelled exactly as shown
- Use the fewest primitives needed — prefer 1-2 over 3-4
- Focus on core meaning, not connotation
- If a word is a function word (the, a, is) use the closest primitive or ["OTHER"]
- Output ONLY valid JSON, no explanation outside the JSON

Example input: thirsty
Example output: {{"word": "thirsty", "primitives": ["SELF", "WANT", "WATER"], "note": "desire for liquid"}}
"""

print('System prompt ready.')
print(f'Prompt length: {len(SYSTEM_PROMPT)} chars')

In [ ]:
# --- Load word list ---
N_WORDS = 500  # start with top 500; increase to 5000+ once validated

words = top_n_list('en', N_WORDS)

# Filter: skip pure numbers, single chars (except I/a)
words = [w for w in words if len(w) > 1 or w.lower() in ('i', 'a')]
words = [w for w in words if not w.isdigit()]

print(f'Word list: {len(words)} words')
print(f'Sample: {words[:20]}')

In [ ]:
# --- API client ---
# Needs ANTHROPIC_API_KEY in environment
# Set it: export ANTHROPIC_API_KEY=sk-ant-...

api_key = os.environ.get('ANTHROPIC_API_KEY')
if not api_key:
    raise EnvironmentError('Set ANTHROPIC_API_KEY environment variable before running.')

client = anthropic.Anthropic(api_key=api_key)
print('Client ready.')

In [ ]:
# --- Decompose words in batches ---
# Sends 10 words per API call to reduce cost; system prompt is cached.

BATCH_SIZE = 10
OUTPUT_FILE = Path('/Users/ryanfunk/primatap_dictionary.json')

# Load existing results if resuming
if OUTPUT_FILE.exists():
    dictionary = json.loads(OUTPUT_FILE.read_text())
    print(f'Resuming — {len(dictionary)} words already processed')
else:
    dictionary = {}

# Filter to unprocessed words
remaining = [w for w in words if w not in dictionary]
batches = [remaining[i:i+BATCH_SIZE] for i in range(0, len(remaining), BATCH_SIZE)]

print(f'{len(remaining)} words to process in {len(batches)} batches')

In [ ]:
def decompose_batch(word_batch):
    """Send a batch of words to Claude, return list of parsed results."""
    user_msg = 'Decompose each of these words into primitives. Return a JSON array of objects.\n\n'
    user_msg += '\n'.join(word_batch)

    response = client.messages.create(
        model='claude-haiku-4-5-20251001',  # fast + cheap for bulk dictionary work
        max_tokens=1024,
        system=[
            {
                'type': 'text',
                'text': SYSTEM_PROMPT,
                'cache_control': {'type': 'ephemeral'}  # cache the system prompt
            }
        ],
        messages=[{'role': 'user', 'content': user_msg}]
    )

    raw = response.content[0].text.strip()

    # Parse JSON — handle both array and single object responses
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            parsed = [parsed]
    except json.JSONDecodeError:
        # Try extracting JSON array from response
        start = raw.find('[')
        end = raw.rfind(']') + 1
        if start >= 0 and end > start:
            parsed = json.loads(raw[start:end])
        else:
            print(f'  Parse failed for batch: {word_batch}')
            return []

    # Validate primitives
    results = []
    for item in parsed:
        valid_prims = [p for p in item.get('primitives', []) if p in PRIMITIVE_SET]
        if valid_prims:
            item['primitives'] = valid_prims
            results.append(item)

    return results


# --- Run ---
errors = []

for i, batch in enumerate(batches):
    print(f'Batch {i+1}/{len(batches)}: {batch}', end=' ... ')
    try:
        results = decompose_batch(batch)
        for item in results:
            dictionary[item['word']] = {
                'primitives': item['primitives'],
                'note': item.get('note', '')
            }
        print(f'got {len(results)}')
    except Exception as e:
        print(f'ERROR: {e}')
        errors.append((batch, str(e)))

    # Save after every batch
    OUTPUT_FILE.write_text(json.dumps(dictionary, indent=2))
    time.sleep(0.3)  # gentle rate limiting

print(f'\nDone. {len(dictionary)} words in dictionary. {len(errors)} errors.')

In [ ]:
# --- Inspect results ---
import random

print('=== Sample entries ===')
sample = random.sample(list(dictionary.items()), min(20, len(dictionary)))
for word, data in sorted(sample, key=lambda x: len(x[1]['primitives'])):
    prims = ' + '.join(data['primitives'])
    print(f'  {word:<15} → {prims:<35}  ({data["note"]})')

In [ ]:
# --- Statistics ---
import matplotlib.pyplot as plt
from collections import Counter

# Primitive usage frequency
prim_counts = Counter()
for data in dictionary.values():
    for p in data['primitives']:
        prim_counts[p] += 1

# Sequence length distribution
lengths = [len(data['primitives']) for data in dictionary.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Primitive frequency
prims_sorted = sorted(prim_counts.items(), key=lambda x: -x[1])
ax1.barh([p for p,_ in prims_sorted], [c for _,c in prims_sorted], color='steelblue')
ax1.set_title('Primitive Usage Frequency\n(validates Huffman chord assignment)')
ax1.set_xlabel('Words using this primitive')
ax1.invert_yaxis()

# Sequence lengths
length_counts = Counter(lengths)
ax2.bar(length_counts.keys(), length_counts.values(), color='darkorange')
ax2.set_title('Primitive Sequence Length Distribution')
ax2.set_xlabel('Number of primitives per word')
ax2.set_ylabel('Count')
ax2.set_xticks(sorted(length_counts.keys()))

plt.tight_layout()
plt.savefig('/Users/ryanfunk/dictionary_stats.png', dpi=150)
plt.show()

print(f'Average primitives per word: {sum(lengths)/len(lengths):.2f}')
print(f'Most used primitive: {prims_sorted[0][0]} ({prims_sorted[0][1]} words)')
print(f'Least used primitive: {prims_sorted[-1][0]} ({prims_sorted[-1][1]} words)')

In [ ]:
# --- Export as CSV for easy reading ---
import csv

csv_path = '/Users/ryanfunk/primatap_dictionary.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['word', 'primitives', 'chord_sequence', 'note'])
    for word, data in sorted(dictionary.items()):
        prims = data['primitives']
        chord_seq = ' → '.join(prims)
        writer.writerow([word, '|'.join(prims), chord_seq, data['note']])

print(f'CSV saved: {csv_path}')
print(f'JSON saved: {OUTPUT_FILE}')
print(f'Total entries: {len(dictionary)}')